# Pre-Modeling Summary

## 7.1 Dataset Overview

The final dataset used for modeling contains **920 patients** and 
**11 features + 1 target variable**, after all preprocessing steps 
described in this EDA.

| | Value |
|---|---|
| Total patients | 920 |
| Features | 11 |
| Target variable | `target` (binary) |
| Missing values | 0 (after imputation) |
| Positive class (CAD) | 509 (55.3%) |
| Negative class (No CAD) | 411 (44.7%) |

---

## 7.2 Features Selected for Modeling

| Rank | Feature | Type | Expected Weight | Rationale |
|------|---------|------|-----------------|-----------|
| 1 | `exang` | Categorical | High | 83.7% CAD prevalence when positive |
| 2 | `thalch` | Continuous | High | Clearest distribution separation between groups |
| 3 | `cp` | Categorical | High | 79.0% CAD in asymptomatic group |
| 4 | `oldpeak` | Continuous | High | Direct ischemia measurement (r=0.40 with target) |
| 5 | `slope` | Categorical | High | ~77% CAD in flat/downsloping categories |
| 6 | `age` | Continuous | Medium | Strong epidemiological trend after age 45 |
| 7 | `sex` | Categorical | Medium | Significant prevalence difference (63% male vs 26% female) |
| 8 | `restecg` | Categorical | Moderate | Consistent trend but weak discrimination at rest |
| 9 | `trestbps` | Continuous | Low | Heavy overlap between disease/no-disease groups |
| 10 | `chol` | Continuous | Low | Heavy overlap; limited by absence of LDL/HDL ratio |
| 11 | `fbs` | Categorical | Low | Modest separation (49% vs 68%); limited positive cases (n=138) |

*This ranking will be validated against SHAP values after model training.*

---

## 7.3 Excluded Variables

| Variable | Reason |
|----------|--------|
| `ca` | 66.4% missing — fluoroscopy not performed across institutions |
| `thal` | 52.8% missing — nuclear stress test not performed across institutions |
| `dataset` | Data leakage — institution of origin must not predict disease |
| `id` | Row identifier, no clinical meaning |
| `num` | Original multiclass target, replaced by binary `target` |

---

## 7.4 Preprocessing Decisions

| Step | Decision | Justification |
|------|----------|---------------|
| Target binarization | `num` > 0 → 1 | Clinical threshold: ≥50% stenosis = significant CAD (Detrano et al., 1989) |
| Cholesterol zeros | Recoded as NaN | Physiologically impossible; encoded missingness |
| Implausible values | Recoded as NaN | trestbps=0, oldpeak<-2, chol<100 |
| Categorical encoding | OrdinalEncoder | Required for IterativeImputer numeric input |
| Missing imputation | IterativeImputer (max_iter=10) | Predicts each missing value using all other variables |
| Outlier treatment | Retained if clinically plausible | e.g. chol=564 (familial hypercholesterolemia), trestbps=200 (hypertensive crisis) |

---

## 7.5 Evaluation Strategy

**Validation:** Stratified K-Fold Cross-Validation (k=5)
Each fold preserves the 44.7%/55.3% class distribution of the 
full dataset.

**Hold-out set:** 20% of data reserved before any model development 
for final evaluation of the best-performing model.

**Primary metrics:**
- ROC-AUC — discrimination between classes
- F1-Score — balance between precision and recall
- Precision-Recall AUC — performance on positive class
- Calibration (Brier Score) — reliability of predicted probabilities

**Clinical cost consideration:** False negatives (missed CAD) carry 
higher clinical cost than false positives (unnecessary further testing). 
Recall will be monitored alongside precision.

**Fairness evaluation:** Model performance will be reported separately 
by sex after training, given the underrepresentation of female patients 
(n=194, 21%) and their substantially lower CAD prevalence (25.8% vs 63.2%).

---

## 7.6 Known Limitations

1. **Referral bias:** All 920 patients were referred for coronary 
   angiography — CAD prevalence (55.3%) far exceeds the general 
   population (~6-7%). The model is not suitable for general 
   population screening without recalibration.

2. **Institutional heterogeneity:** CAD prevalence varies substantially 
   across institutions (Cleveland 45.7% → Switzerland 93.5%). The model 
   learns from a heterogeneous population and may not generalize well 
   to any single institution's patient profile.

3. **Missing stress test data:** The strongest predictors (`thalch`, 
   `exang`, `oldpeak`, `slope`) all require exercise stress testing — 
   not always performed in clinical practice. Model performance may 
   degrade significantly in patients without stress test data.

4. **Cholesterol limitation:** Only total cholesterol is available. 
   The LDL/HDL ratio is clinically more informative but absent from 
   this dataset.

5. **Female underrepresentation:** Only 194 female patients (21%) 
   with 25.8% CAD prevalence. Model performance for female patients 
   should be interpreted cautiously.